# 🏪 Module 4 & 5: Seller Reliability + Return Risk
**ECommerce Product Trust Assessor**

Uses the **Olist Brazilian E-Commerce Dataset** to train:
- XGBoost Seller Reliability Score
- LightGBM Return Risk Predictor

**Dataset:** https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

In [ ]:
!pip install -q kaggle xgboost lightgbm shap scikit-learn pandas matplotlib seaborn joblib

In [ ]:
# ── Download Olist Dataset ───────────────────────────────────────────────────
# Option 1: Kaggle API
# !kaggle datasets download -d olistbr/brazilian-ecommerce --unzip

# Option 2: Upload manually via Colab file picker
# from google.colab import files
# uploaded = files.upload()

# For demo: generate synthetic data matching Olist schema
import numpy as np
import pandas as pd

np.random.seed(42)
N = 10_000

sellers = pd.DataFrame({
    'seller_id': [f'seller_{i}' for i in range(500)],
    'avg_review_score': np.random.beta(5, 2, 500) * 4 + 1,
    'total_orders': np.random.exponential(200, 500).astype(int) + 10,
    'cancellation_rate': np.random.beta(1, 20, 500),
    'avg_delivery_days': np.random.gamma(3, 3, 500) + 3,
    'late_delivery_rate': np.random.beta(1, 10, 500),
})

print('Sellers dataset:')
print(sellers.describe())

In [ ]:
# ── Feature Engineering: Seller Reliability ──────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

def build_seller_features(df):
    features = df.copy()

    # Normalize rating to 0-1
    features['rating_norm'] = (features['avg_review_score'] - 1) / 4

    # Log-scale order volume
    features['log_orders'] = np.log1p(features['total_orders'])

    # Delivery score: lower days = better
    features['delivery_score'] = 1 - np.clip(
        features['avg_delivery_days'] / 30, 0, 1
    )

    # Reliability score (pseudo-label)
    features['reliability_score'] = (
        features['rating_norm'] * 0.4 +
        (1 - features['cancellation_rate']) * 0.3 +
        features['delivery_score'] * 0.2 +
        np.clip(features['log_orders'] / 8, 0, 1) * 0.1
    ) * 100

    # Binary label: reliable if score > 70
    features['is_reliable'] = (features['reliability_score'] > 70).astype(int)

    return features

sellers = build_seller_features(sellers)
print('Reliability distribution:')
print(sellers['is_reliable'].value_counts(normalize=True))

In [ ]:
# ── Train Seller Reliability Model ───────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
import xgboost as xgb
import shap
import joblib

SELLER_FEATURES = [
    'rating_norm', 'log_orders', 'cancellation_rate',
    'delivery_score', 'late_delivery_rate'
]

X = sellers[SELLER_FEATURES]
y = sellers['is_reliable']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

xgb_seller = xgb.XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.05,
    subsample=0.8, random_state=42, eval_metric='logloss'
)
xgb_seller.fit(X_train, y_train)

print(classification_report(y_test, xgb_seller.predict(X_test)))
print(f'ROC-AUC: {roc_auc_score(y_test, xgb_seller.predict_proba(X_test)[:,1]):.4f}')

In [ ]:
# ── Train Return Risk Model ───────────────────────────────────────────────────
import lightgbm as lgb

# Simulate order-level data for return risk
N_ORDERS = 30_000
orders = pd.DataFrame({
    'price': np.random.exponential(50, N_ORDERS) + 5,
    'category_risk': np.random.choice([0, 0.3, 0.6, 1.0], N_ORDERS),
    'seller_reliability': np.random.uniform(30, 100, N_ORDERS),
    'sentiment_score': np.random.uniform(20, 100, N_ORDERS),
    'product_rating': np.random.choice([1,2,3,4,5], N_ORDERS,
                                        p=[0.05, 0.1, 0.15, 0.35, 0.35]),
})

# Pseudo-label return probability
return_prob = (
    0.05 +
    orders['category_risk'] * 0.15 +
    np.clip((orders['price'] - 50) / 500, 0, 0.1) +
    np.clip((3 - orders['product_rating']) * 0.05, 0, 0.15) +
    np.clip((50 - orders['sentiment_score']) / 200, 0, 0.1)
)
orders['returned'] = (np.random.random(N_ORDERS) < return_prob).astype(int)

RETURN_FEATURES = ['price', 'category_risk', 'seller_reliability',
                   'sentiment_score', 'product_rating']

Xr = orders[RETURN_FEATURES]
yr = orders['returned']

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    Xr, yr, test_size=0.2, random_state=42
)

lgbm_return = lgb.LGBMClassifier(
    n_estimators=300, num_leaves=31,
    learning_rate=0.05, random_state=42
)
lgbm_return.fit(Xr_train, yr_train)

print('Return Risk Model:')
print(classification_report(yr_test, lgbm_return.predict(Xr_test)))

In [ ]:
# ── SHAP + Save ───────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import os

explainer = shap.TreeExplainer(xgb_seller)
shap_vals = explainer.shap_values(X_test)

shap.summary_plot(shap_vals, X_test, feature_names=SELLER_FEATURES, show=False)
plt.title('Seller Reliability — SHAP Feature Importance')
plt.tight_layout()
plt.savefig('shap_seller.png', dpi=150)
plt.show()

os.makedirs('models/seller', exist_ok=True)
os.makedirs('models/return_risk', exist_ok=True)

joblib.dump(xgb_seller, 'models/seller/xgb_seller.joblib')
joblib.dump(lgbm_return, 'models/return_risk/lgbm_return_risk.joblib')
joblib.dump(SELLER_FEATURES, 'models/seller/feature_cols.joblib')
joblib.dump(RETURN_FEATURES, 'models/return_risk/feature_cols.joblib')

print('✅ Models saved.')